In [57]:
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis
from tqdm import tqdm
from pathlib import Path
from data_loader import build_complete_dataset
from window import create_record_windows
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix)
from sklearn.model_selection import ParameterGrid
import json

In [58]:
def extract_gyro_features(window_df, fs):
    features = {}
    columns = [column for column in window_df.columns if column != "Time" ]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
            signal = data[:, i]
            features[f"gyro_{col}_mean"] = np.mean(signal)
            features[f"gyro_{col}_std"] = np.std(signal)
            features[f"gyro_{col}_rms"] = np.sqrt(np.mean(signal**2))
            features[f"gyro_{col}_max"] = np.max(signal)
            features[f"gyro_{col}_min"] = np.min(signal)
            features[f"gyro_{col}_ptp"] = np.ptp(signal)
            features[f"gyro_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
            features[f"gyro_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
            fft_vals = np.abs(np.fft.rfft(signal))
            freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
            features[f"gyro_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features

In [59]:
def extract_acc_features(window_df, fs):
    features = {}
    columns = [column for column in window_df.columns if column != "Time" ]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
            signal = data[:, i]
            features[f"acc_{col}_mean"] = np.mean(signal)
            features[f"acc_{col}_std"] = np.std(signal)
            features[f"acc_{col}_rms"] = np.sqrt(np.mean(signal**2))
            features[f"acc_{col}_max"] = np.max(signal)
            features[f"acc_{col}_min"] = np.min(signal)
            features[f"acc_{col}_ptp"] = np.ptp(signal)
            features[f"acc_{col}_skew"] = skew(signal) if len(signal) > 3 else 0
            features[f"acc_{col}_kurtosis"] = kurtosis(signal) if len(signal) > 3 else 0
            fft_vals = np.abs(np.fft.rfft(signal))
            freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
            features[f"acc_{col}_fft_energy"] = np.sum(fft_vals**2)
            features[f"acc_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features

In [60]:
def extract_mic_features(window_df, fs):
    features = {}
    columns = [column for column in window_df.columns if column != "Time" ]
    data = window_df.drop(columns=["Time"], errors="ignore").values
    for i, col in enumerate(columns):
            signal = data[:, i]
            features[f"mic_{col}_mean"] = np.mean(signal)
            features[f"mic_{col}_std"] = np.std(signal)
            features[f"mic_{col}_rms"] = np.sqrt(np.mean(signal**2))
            features[f"mic_{col}_max"] = np.max(signal)
            features[f"mic_{col}_min"] = np.min(signal)
            features[f"mic_{col}_ptp"] = np.ptp(signal)     
            fft_vals = np.abs(np.fft.rfft(signal))
            freqs = np.fft.rfftfreq(len(signal), d=1 / fs)
            features[f"mic_{col}_fft_peak_freq"] = freqs[np.argmax(fft_vals)]
    return features

In [61]:
def process_record_features(record, create_record_windows_func):
    acc_windows, gyro_windows, mic_windows = create_record_windows_func(record)
    
    record_features = []
    min_len = min(len(acc_windows), len(gyro_windows), len(mic_windows))
    
    for i in range(min_len):
        features = {}
        features.update(extract_acc_features(acc_windows[i], fs=6700))
        features.update(extract_gyro_features(gyro_windows[i], fs=6700))
        features.update(extract_mic_features(mic_windows[i], fs=16000))
        
        features["segment_id"] = record.metadata.get("segment_id")
        features["split_label"] = record.metadata.get("split_label")
        features["anomaly_label"] = record.metadata.get("anomaly_label")
        features["domain_shift_op"] = record.metadata.get("domain_shift_op")
        features["domain_shift_env"] = record.metadata.get("domain_shift_env")
        record_features.append(features)
        
    return pd.DataFrame(record_features)

In [62]:
def build_feature_dataset(dataset, create_record_windows_func):
    results = []
    for record in tqdm(dataset, desc="Ekstrakcja cech"):
        df_feats = process_record_features(record, create_record_windows_func)
        results.append(df_feats)
    return pd.concat(results, ignore_index=True)

In [63]:
path = Path("../data")
train_path = path / "X_train.csv"
test_path = path / "X_test.csv"

if train_path.exists() and test_path.exists():
    X_train_df = pd.read_csv(train_path)
    X_test_df = pd.read_csv(test_path)
else:
    df = build_complete_dataset()
    X_train_df = build_feature_dataset(df[0], create_record_windows)
    X_test_df = build_feature_dataset(df[1], create_record_windows)
    X_train_df.to_csv(train_path, index=False)
    X_test_df.to_csv(test_path, index=False)

print("Train:", X_train_df.shape)
print("Test :", X_test_df.shape)

Train: (102984, 69)
Test : (25984, 69)


In [64]:
X_train_source = X_train_df[X_train_df["split_label"] == "Normal_Source_Train"].copy()
X_val = X_test_df[X_test_df["split_label"].isin(["Normal_Source_Test", "Anomaly_Source_Test"])].copy() # zbior walidacyjny
X_test = X_test_df[X_test_df["split_label"].isin(["Normal_Target_Test", "Anomaly_Target_Test"])].copy() # to nasz zbiór testowy

In [65]:
acc_features = [col for col in X_train_df.columns if col.startswith("acc_")]
gyro_features = [col for col in X_train_df.columns if col.startswith("gyro_")]
mic_features = [col for col in X_train_df.columns if col.startswith("mic_")]

In [66]:
X_train_source_acc = X_train_source[acc_features].fillna(0)
X_train_source_mic = X_train_source[mic_features].fillna(0)
X_train_source_gyro = X_train_source[gyro_features].fillna(0)

X_val_acc = X_val[acc_features].fillna(0)
X_val_mic = X_val[mic_features].fillna(0)
X_val_gyro = X_val[gyro_features].fillna(0)

X_test_acc = X_test[acc_features].fillna(0)
X_test_mic = X_test[mic_features].fillna(0)
X_test_gyro = X_test[gyro_features].fillna(0)


In [67]:
def find_best_threshold(y_true, scores):
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)
    thresholds = np.percentile(scores, np.linspace(0, 100, 200))
    best_threshold = thresholds[0]
    best_f1 = -1
    for threshold in thresholds:
        y_pred = (scores >= threshold).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

In [68]:
def tune_isolation_forest(
    X_train,
    X_val,
    val_df,
    sensor_type,
    param_grid):

    aggregation_methods = ["max", "mean", "median" ]

    results = []

    y_val = (val_df.groupby("segment_id")["anomaly_label"].first().eq("loosescrewsA").astype(int))

    for params in ParameterGrid(param_grid):
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        model = IsolationForest(
            n_estimators=params["n_estimators"],
            max_samples=params["max_samples"],
            max_features=params["max_features"],
            bootstrap=params["bootstrap"],
            contamination="auto",
            random_state=42,
            n_jobs=-1)

        model.fit(X_train_scaled)

        val_scores = -model.score_samples(X_val_scaled)

        val_results = pd.DataFrame({
            "segment_id": val_df["segment_id"].values,
            "score": val_scores})


        for aggregation in aggregation_methods:
            if aggregation == "max":
                segment_scores = (val_results.groupby("segment_id")["score"].max().reset_index())
            elif aggregation == "mean":
                segment_scores = (val_results.groupby("segment_id")["score"].mean().reset_index())
            elif aggregation == "median":
                segment_scores = (val_results.groupby("segment_id")["score"].median().reset_index())

            segment_scores = segment_scores.rename(columns={"score": "score"})
            segment_scores = segment_scores.merge(y_val.rename("y_true"), left_on="segment_id", right_index=True, how="inner")
            threshold, best_f1 = find_best_threshold(segment_scores["y_true"].values, segment_scores["score"].values)

            y_pred = (segment_scores["score"] >= threshold).astype(int)
            precision = precision_score(segment_scores["y_true"], y_pred, zero_division=0)
            recall = recall_score(segment_scores["y_true"], y_pred, zero_division=0)
            accuracy = accuracy_score(segment_scores["y_true"], y_pred)
            roc_auc = roc_auc_score(segment_scores["y_true"], segment_scores["score"])
            results.append({
                "sensor": sensor_type,
                **params,
                "aggregation": aggregation,
                "threshold": threshold,
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "ROC AUC": roc_auc,
                "F1": best_f1
            })


    results_df = pd.DataFrame(results)
    best_row = results_df.loc[results_df["F1"].idxmax()]

    best_params = {
        "n_estimators": int(best_row["n_estimators"]),
        "max_samples": best_row["max_samples"],
        "max_features": best_row["max_features"],
        "bootstrap": bool(best_row["bootstrap"]),
    }

    best_threshold = best_row["threshold"]
    best_aggregation = best_row["aggregation"]

    print(f"Sensor: {sensor_type}")

    print("Parameters:")
    print(best_params)

    print(f"Aggregation: {best_aggregation}")
    print(f"Threshold  : {best_threshold:.6f}")
    print(f"Accuracy   : {best_row['Accuracy']:.4f}")
    print(f"Precision  : {best_row['Precision']:.4f}")
    print(f"Recall     : {best_row['Recall']:.4f}")
    print(f"ROC AUC    : {best_row['ROC AUC']:.4f}")
    print(f"F1         : {best_row['F1']:.4f}")

    print("All configurations:")
    print(results_df.sort_values("F1", ascending=False).head(10))
    return (best_params, best_threshold, best_aggregation, results_df)

In [69]:
param_grid = {
    "n_estimators": [50, 100, 200, 300, 400],
    "max_samples": [256, 512, 1024, 2048],
    "max_features": [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "bootstrap": [False, True]
}

In [70]:
best_params_acc, threshold_acc, best_acc_aggregation, tuning_acc = tune_isolation_forest(
    X_train_source_acc,
    X_val_acc,
    X_val,
    "ACC source",
    param_grid
)

Sensor: ACC source
Parameters:
{'n_estimators': 50, 'max_samples': 1024, 'max_features': 0.8, 'bootstrap': False}
Aggregation: max
Threshold  : 0.526835
Accuracy   : 0.7716
Precision  : 0.7203
Recall     : 0.8879
ROC AUC    : 0.7525
F1         : 0.7954
All configurations:
         sensor  bootstrap  max_features  max_samples  n_estimators  \
210  ACC source      False           0.8         1024            50   
465  ACC source       True           0.6         2048            50   
405  ACC source       True           0.5         2048            50   
30   ACC source      False           0.5         1024            50   
420  ACC source       True           0.6          256            50   
234  ACC source      False           0.8         2048           300   
60   ACC source      False           0.6          256            50   
345  ACC source      False           1.0         2048            50   
63   ACC source      False           0.6          256           100   
105  ACC source  

In [71]:
best_params_mic, threshold_mic, best_mic_aggregation, tuning_mic = tune_isolation_forest(
    X_train_source_mic,
    X_val_mic,
    X_val,
    "MIC source",
    param_grid
)

Sensor: MIC source
Parameters:
{'n_estimators': 50, 'max_samples': 2048, 'max_features': 0.9, 'bootstrap': False}
Aggregation: median
Threshold  : 0.366067
Accuracy   : 0.5948
Precision  : 0.5545
Recall     : 0.9655
ROC AUC    : 0.6236
F1         : 0.7044
All configurations:
         sensor  bootstrap  max_features  max_samples  n_estimators  \
287  MIC source      False           0.9         2048            50   
53   MIC source      False           0.5         2048           200   
212  MIC source      False           0.8         1024            50   
677  MIC source       True           1.0          512            50   
557  MIC source       True           0.8          512            50   
203  MIC source      False           0.8          512           200   
197  MIC source      False           0.8          512            50   
311  MIC source      False           1.0          256           300   
647  MIC source       True           0.9         2048            50   
32   MIC sourc

In [72]:
best_params_gyro, threshold_gyro, best_gyro_aggregation, tuning_gyro = tune_isolation_forest(
    X_train_source_gyro,
    X_val_gyro,
    X_val,
    "GYRO source",
    param_grid
)

Sensor: GYRO source
Parameters:
{'n_estimators': 50, 'max_samples': 2048, 'max_features': 0.8, 'bootstrap': False}
Aggregation: median
Threshold  : 0.397232
Accuracy   : 0.7716
Precision  : 0.6957
Recall     : 0.9655
ROC AUC    : 0.7626
F1         : 0.8087
All configurations:
          sensor  bootstrap  max_features  max_samples  n_estimators  \
227  GYRO source      False           0.8         2048            50   
230  GYRO source      False           0.8         2048           100   
287  GYRO source      False           0.9         2048            50   
536  GYRO source       True           0.7         2048           300   
239  GYRO source      False           0.8         2048           400   
167  GYRO source      False           0.7         2048            50   
359  GYRO source      False           1.0         2048           400   
533  GYRO source       True           0.7         2048           200   
539  GYRO source       True           0.7         2048           400   
347

In [73]:
final_params = {
    "ACC": {
        "n_estimators": int(best_params_acc["n_estimators"]),
        "max_samples": int(best_params_acc["max_samples"]),
        "max_features": float(best_params_acc["max_features"]),
        "bootstrap": bool(best_params_acc["bootstrap"]),
        "threshold": float(threshold_acc),
        "aggregation": best_acc_aggregation
    },

    "MIC": {
        "n_estimators": int(best_params_mic["n_estimators"]),
        "max_samples": int(best_params_mic["max_samples"]),
        "max_features": float(best_params_mic["max_features"]),
        "bootstrap": bool(best_params_mic["bootstrap"]),
        "threshold": float(threshold_mic),
        "aggregation": best_mic_aggregation
    },

    "GYRO": {
        "n_estimators": int(best_params_gyro["n_estimators"]),
        "max_samples": int(best_params_gyro["max_samples"]),
        "max_features": float(best_params_gyro["max_features"]),
        "bootstrap": bool(best_params_gyro["bootstrap"]),
        "threshold": float(threshold_gyro),
        "aggregation": best_gyro_aggregation
    }
}

params_path = path / "final_if_params.json"

with open(params_path, "w") as f:
    json.dump(final_params, f, indent=4)

print(f"parameters saved to: {params_path}")

parameters saved to: ..\data\final_if_params.json
